In [1]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPU disponibles:", tf.config.list_physical_devices('GPU'))

TensorFlow: 2.20.0
GPU disponibles: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
import os
import zipfile

archivo_zip = "/content/Enfermedades Cacao.zip"
destino = "/content/dataset_cacao"

with zipfile.ZipFile(archivo_zip, 'r') as zip_ref:
    zip_ref.extractall(destino)

print("Dataset descomprimido correctamente.")

Dataset descomprimido correctamente.


In [4]:
import os

ruta_dataset = "/content/dataset_cacao"

for raiz, carpetas, archivos in os.walk(ruta_dataset):
    nivel = raiz.replace(ruta_dataset, "").count(os.sep)
    sangria = "    " * nivel

    print(f"{sangria}📁 {os.path.basename(raiz)}/")

    if archivos:
        print(f"{sangria}   🖼️ {len(archivos)} archivos")

📁 dataset_cacao/
    📁 Fito/
       🖼️ 215 archivos
    📁 Monilia/
       🖼️ 211 archivos
    📁 Sana/
       🖼️ 201 archivos


In [7]:
import os
import random
import shutil

# Para que siempre obtengamos la misma selección
random.seed(42)

# IMPORTANTE:
# Ajusta esta ruta si en tu resultado anterior apareció otro nombre
ruta_original = "/content/dataset_cacao"

ruta_muestra = "/content/muestra_cacao"

clases = ["Sana", "Monilia", "Fito"]
cantidad_por_clase = 100

# Extensiones de imagen aceptadas
extensiones = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

for clase in clases:
    origen = os.path.join(ruta_original, clase)
    destino = os.path.join(ruta_muestra, clase)

    os.makedirs(destino, exist_ok=True)

    imagenes = [
        archivo for archivo in os.listdir(origen)
        if archivo.lower().endswith(extensiones)
    ]

    seleccionadas = random.sample(imagenes, cantidad_por_clase)

    for imagen in seleccionadas:
        shutil.copy2(
            os.path.join(origen, imagen),
            os.path.join(destino, imagen)
        )

    print(f"{clase}: {len(seleccionadas)} imágenes seleccionadas")

Sana: 100 imágenes seleccionadas
Monilia: 100 imágenes seleccionadas
Fito: 100 imágenes seleccionadas


In [6]:
import os

for raiz, carpetas, archivos in os.walk("/content/dataset_cacao"):
    print(raiz)

/content/dataset_cacao
/content/dataset_cacao/Fito
/content/dataset_cacao/Monilia
/content/dataset_cacao/Sana


In [8]:
import os
import random
import shutil

random.seed(42)

ruta_muestra = "/content/muestra_cacao"
ruta_dividida = "/content/cacao_split"

clases = ["Sana", "Monilia", "Fito"]
extensiones = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

for clase in clases:

    ruta_clase = os.path.join(ruta_muestra, clase)

    imagenes = [
        archivo for archivo in os.listdir(ruta_clase)
        if archivo.lower().endswith(extensiones)
    ]

    # Mezclar aleatoriamente
    random.shuffle(imagenes)

    # 70 / 15 / 15
    train = imagenes[:70]
    val = imagenes[70:85]
    test = imagenes[85:100]

    divisiones = {
        "train": train,
        "validation": val,
        "test": test
    }

    for division, lista_imagenes in divisiones.items():

        destino = os.path.join(
            ruta_dividida,
            division,
            clase
        )

        os.makedirs(destino, exist_ok=True)

        for imagen in lista_imagenes:
            shutil.copy2(
                os.path.join(ruta_clase, imagen),
                os.path.join(destino, imagen)
            )

print("División 70/15/15 realizada correctamente.")

División 70/15/15 realizada correctamente.


In [10]:
total_general = 0

for division in ["train", "validation", "test"]:

    print(f"\n{division.upper()}")
    total_division = 0

    for clase in clases:

        ruta = os.path.join(
            ruta_dividida,
            division,
            clase
        )

        cantidad = len([
            archivo for archivo in os.listdir(ruta)
            if archivo.lower().endswith(extensiones)
        ])

        print(f"{clase}: {cantidad}")
        total_division += cantidad

    print(f"Total {division}: {total_division}")
    total_general += total_division

print("\n---------------------")
print(f"TOTAL GENERAL: {total_general}")


TRAIN
Sana: 70
Monilia: 70
Fito: 70
Total train: 210

VALIDATION
Sana: 15
Monilia: 15
Fito: 15
Total validation: 45

TEST
Sana: 15
Monilia: 15
Fito: 15
Total test: 45

---------------------
TOTAL GENERAL: 300


In [11]:
import tensorflow as tf

# Rutas
ruta_train = "/content/cacao_split/train"
ruta_val = "/content/cacao_split/validation"
ruta_test = "/content/cacao_split/test"

# Parámetros para EfficientNet-B1
IMG_SIZE = (240, 240)
BATCH_SIZE = 16
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    ruta_train,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=True,
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    ruta_val,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    ruta_test,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

Found 210 files belonging to 3 classes.
Found 45 files belonging to 3 classes.
Found 45 files belonging to 3 classes.


In [12]:
print("Clases detectadas:", train_ds.class_names)

Clases detectadas: ['Fito', 'Monilia', 'Sana']


In [13]:
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB1
from tensorflow.keras import layers, Model

# Modelo base EfficientNet-B1
base_model = EfficientNetB1(
    weights="imagenet",
    include_top=False,
    input_shape=(240, 240, 3)
)

# Para este primer entrenamiento congelamos EfficientNet-B1
base_model.trainable = False

# Nueva cabeza de clasificación para nuestras 3 clases
inputs = tf.keras.Input(shape=(240, 240, 3))

x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)

outputs = layers.Dense(
    3,
    activation="softmax"
)(x)

model = Model(inputs, outputs)

model.summary()

27018416/27018416 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 240, 240, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb1 (Functional)     │ (None, 8, 8, 1280)     │     6,575,239 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 3)              │         3,843 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,579,082 (25.10 MB)

 Trainable params: 3,843 (15.01 KB)

 Non-trainable params: 6,575,239 (25.08 MB)

In [14]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("Modelo compilado correctamente.")

Modelo compilado correctamente.


In [15]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

In [16]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

Epoch 1/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 108s 5s/step - accuracy: 0.3238 - loss: 1.1746 - val_accuracy: 0.4667 - val_loss: 0.9950
Epoch 2/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 24s 2s/step - accuracy: 0.5000 - loss: 0.9930 - val_accuracy: 0.6000 - val_loss: 0.8877
Epoch 3/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 40s 2s/step - accuracy: 0.5857 - loss: 0.8802 - val_accuracy: 0.6222 - val_loss: 0.8199
Epoch 4/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 40s 2s/step - accuracy: 0.6810 - loss: 0.7797 - val_accuracy: 0.6667 - val_loss: 0.7752
Epoch 5/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 21s 2s/step - accuracy: 0.6952 - loss: 0.7601 - val_accuracy: 0.6889 - val_loss: 0.7513
Epoch 6/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 22s 2s/step - accuracy: 0.7524 - loss: 0.6839 - val_accuracy: 0.6889 - val_loss: 0.7275
Epoch 7/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 42s 2s/step - accuracy: 0.7238 - loss: 0.6529 - val_accuracy: 0.6889 - val_loss: 0.7159
Epoch 8/10
14/14 ━━━━━━━━━━━━━━━━━━━━ 24s 2s/step - accuracy: 0.7524 - loss: 0.6121 - val_accuracy: 0.6889 - val_loss

In [1]:
model.save("/content/efficientnetb1_cacao_preliminar.keras")

print("Modelo guardado correctamente.")

NameError: name 'model' is not defined